# 00 | Build the dataset (run once)

**Question:** Do NCAA performance statistics add predictive value beyond NBA Draft Combine measurements when predicting first-three-season NBA scoring?

This notebook fetches NCAA **player-game** Parquet files, aggregates each player's *last pre-combine college season*, matches NCAA players conservatively by normalized name **and combine year**, and produces exactly **two core files**:

- `data/predraft_features.csv`: combine measurements + NCAA stats (one row per NBA player ID)
- `data/nba_outcomes.csv`: actual NBA points, games and PPG over the three post-combine seasons

The two files are separate to prevent using future NBA performance as a predictor. Each member notebook joins them offline on `player_id`.

**Source:** [SportsDataverse ESPN men's college basketball player box scores](https://github.com/sportsdataverse/sportsdataverse-data/releases/tag/espn_mens_college_basketball_player_boxscores), 2005–2022 season-ending years. NBA measurements/outcomes come from [your existing project's processed CSV](https://github.com/ishaantibdewal/COGS109_NBA/blob/main/output/nba_early_career_ppg.csv), originally constructed using NBA Stats combine and regular-season APIs.

**Limitations:** The match uses exact normalized names within the same year, not an official cross-league ID. Ambiguous and unmatched players are **never assigned someone else's stats**; inspect `data/ncaa_match_audit.csv` before interpretation. This focuses on college players who subsequently appeared in the NBA within three years, not all combine prospects.

In [1]:
# Install once if needed, then restart this cell.
%pip -q install pandas numpy pyarrow requests matplotlib statsmodels scikit-learn scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import re, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import requests

ROOT = Path.cwd()  # Run Jupyter from this package folder or your repository root.
DATA = ROOT / 'data'
CACHE = DATA / 'raw_ncaa'
DATA.mkdir(exist_ok=True); CACHE.mkdir(exist_ok=True)
SOURCE_NBA = ROOT / 'output' / 'nba_early_career_ppg.csv'
F = DATA / 'predraft_features.csv'
Y = DATA / 'nba_outcomes.csv'

# Re-running after the two output CSVs exist makes no network calls.
if F.exists() and Y.exists():
    print('Cached datasets exist. No API/download needed; skip the build cells below.')
else:
    print('First run: source NBA CSV plus NCAA Parquet seasons will be downloaded/cached.')

First run: source NBA CSV plus NCAA Parquet seasons will be downloaded/cached.


## Step 1 | One-time source download

If you cloned your original GitHub repo, the notebook reuses `output/nba_early_career_ppg.csv` **without downloading NBA stats again**. If it is missing, it downloads the same processed CSV from your public repo. NCAA Parquet files are downloaded only when not already cached. This one notebook may make **multiple HTTPS file requests**, but the member notebooks make **zero requests**.

In [3]:
NBA_CSV_URL = 'https://raw.githubusercontent.com/ishaantibdewal/COGS109_NBA/main/output/nba_early_career_ppg.csv'
BASE = ('https://github.com/sportsdataverse/sportsdataverse-data/releases/download/'
        'espn_mens_college_basketball_player_boxscores/player_box_{year}.parquet')
YEARS = range(2005, 2023)  # NCAA season-ending year = combine year

def download_once(url, target, minimum_bytes=1000):
    if target.exists() and target.stat().st_size >= minimum_bytes:
        return
    target.parent.mkdir(parents=True,exist_ok=True)
    tmp = target.with_suffix(target.suffix + '.part')
    try:
        with requests.get(url, stream=True, timeout=(20, 180), headers={'User-Agent':'COGS109-academic-data/1.0'}) as response:
            response.raise_for_status()
            with tmp.open('wb') as h:
                for part in response.iter_content(chunk_size=1024*1024):
                    if part: h.write(part)
        if tmp.stat().st_size < minimum_bytes:
            raise RuntimeError(f'Download too small: {target.name}; check the source link')
        tmp.replace(target)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise

if not (F.exists() and Y.exists()):
    download_once(NBA_CSV_URL, SOURCE_NBA)
    for year in YEARS:
        path = CACHE / f'player_box_{year}.parquet'
        print('Cached' if path.exists() else 'Downloading', year)
        download_once(BASE.format(year=year), path, minimum_bytes=20000)
    print('All sources cached. Subsequent executions use the local files.')

All sources cached. Subsequent executions use the local files.


## Step 2 | Build college-season statistics

Data are **player-games**, so calculate season-level per-game rates using totals and game counts; shooting percentages use attempts as the denominator. Only NBA-relevant pre-combine college seasons (same season-ending year) are used. The raw Parquet files remain cached locally and do not need to be fetched again.

In [4]:
KEEP = ['season','season_type','game_id','athlete_id','athlete_display_name',
        'did_not_play','minutes','points','assists','rebounds','steals','blocks',
        'turnovers','field_goals_made','field_goals_attempted',
        'three_point_field_goals_made','three_point_field_goals_attempted',
        'free_throws_made','free_throws_attempted']

def normalize_name(value):
    if pd.isna(value): return ''
    s = unicodedata.normalize('NFKD', str(value))
    s = ''.join(c for c in s if not unicodedata.combining(c)).lower()
    s = re.sub(r'[^a-z0-9 ]+', ' ', s)
    # Omit suffix tokens, then require exact whole-name agreement and same year.
    tokens = [t for t in s.split() if t not in {'jr','sr','ii','iii','iv'}]
    return ' '.join(tokens)

def aggregate_college_file(path):
    d = pd.read_parquet(path)
    needed = {'season','game_id','athlete_id','athlete_display_name','points','rebounds','assists'}
    missing = needed - set(d.columns)
    if missing: raise ValueError(f'{path.name} missing columns: {sorted(missing)}')
    # ESPN: 2=regular season, 3=postseason. Drop DNP rows if present.
    if 'season_type' in d:
        s = d.season_type.astype(str).str.lower()
        d = d[s.isin(['2','3','regular','postseason'])].copy()
    if 'did_not_play' in d:
        d = d[~d.did_not_play.astype(str).str.lower().isin(['true','1'])]
    d = d.dropna(subset=['athlete_id','game_id','points']).copy()
    if d.empty: return pd.DataFrame()
    # Some feeds repeat a player-game; do not double count.
    d = d.drop_duplicates(['season','athlete_id','game_id'])
    for c in KEEP[6:]:
        if c not in d: d[c] = np.nan
        d[c] = pd.to_numeric(d[c], errors='coerce')
    stats = [c for c in KEEP[6:] if c in d]
    keys=['season','athlete_id','athlete_display_name']
    totals = d.groupby(keys,dropna=False)[stats].sum(min_count=1)
    count = d.groupby(keys,dropna=False).size().rename('ncaa_games')
    g = totals.join(count).reset_index()
    ren = {'minutes':'ncaa_mpg','points':'ncaa_ppg','assists':'ncaa_apg',
           'rebounds':'ncaa_rpg','steals':'ncaa_spg','blocks':'ncaa_bpg',
           'turnovers':'ncaa_tov_pg'}
    for src,dst in ren.items():
        g[dst] = g[src] / g.ncaa_games
    for dst,made,attempted in [
        ('ncaa_fg_pct','field_goals_made','field_goals_attempted'),
        ('ncaa_3p_pct','three_point_field_goals_made','three_point_field_goals_attempted'),
        ('ncaa_ft_pct','free_throws_made','free_throws_attempted')]:
        g[dst] = 100 * g[made] / g[attempted].replace(0, np.nan)
    g['ncaa_name_key'] = g.athlete_display_name.map(normalize_name)
    cols=['season','athlete_id','athlete_display_name','ncaa_name_key','ncaa_games',
          *ren.values(),'ncaa_fg_pct','ncaa_3p_pct','ncaa_ft_pct']
    return g[cols]

if not (F.exists() and Y.exists()):
    parts = [aggregate_college_file(CACHE/f'player_box_{year}.parquet') for year in YEARS]
    college = pd.concat(parts,ignore_index=True)
    college['season'] = pd.to_numeric(college['season'], errors='coerce').astype('Int64')
    print('College player-season rows:',len(college))

College player-season rows: 125777


## Step 3 | Match NCAA players and save **two CSVs**

Only an unambiguous match to an NCAA athlete in the season immediately preceding the NBA combine is accepted. NCAA players without a match are retained in the features dataset with `ncaa_match_status` and missing NCAA columns; all three model notebooks then use the same **matched-only** subset for a fair comparison.

In [5]:
if not (F.exists() and Y.exists()):
    nba = pd.read_csv(SOURCE_NBA)
    required={'player_id','player_name','combine_year','position_group','ppg_3yr',
              'total_games_3yr','total_points_3yr'}
    if required-set(nba): raise ValueError(f'NBA file is missing: {required-set(nba)}')
    if nba.player_id.duplicated().any(): raise ValueError('NBA file contains duplicate player IDs')
    nba['name_key'] = nba.player_name.map(normalize_name)
    nba['combine_year'] = pd.to_numeric(nba.combine_year,errors='raise').astype('int64')
    nba = nba[nba.combine_year.between(2005,2022)].copy()
    # Ignore rare identities that are ambiguous even within a single college season.
    counts = college.groupby(['season','ncaa_name_key']).athlete_id.nunique().rename('n_id').reset_index()
    unique = college.merge(counts, on=['season','ncaa_name_key'],validate='many_to_one')
    unique = unique[(unique.n_id==1) & (unique.ncaa_name_key!='')].drop(columns='n_id')
    # One college identity per name/year; keep only one university-year record if duplicates exist.
    unique=unique.sort_values('ncaa_games',ascending=False).drop_duplicates(['season','ncaa_name_key'])
    merged=nba.merge(unique, left_on=['combine_year','name_key'], right_on=['season','ncaa_name_key'],
                     how='left',validate='one_to_one',indicator=True)
    # A normalized name can occur across multiple NBA players in same cohort. Mark those as ambiguous.
    nba_dup=merged.duplicated(['combine_year','name_key'],keep=False)
    merged['ncaa_match_status']=np.where(nba_dup,'ambiguous_nba_name',
        np.where(merged['_merge'].eq('both'),'matched','unmatched_or_ambiguous_ncaa'))
    ncaa_columns=[c for c in merged if c.startswith('ncaa_') and c not in {'ncaa_match_status','ncaa_name_key'}]
    merged.loc[merged.ncaa_match_status!='matched', ncaa_columns]=np.nan
    feature_cols=['player_id','player_name','combine_year','position_group',
                  'height_in','weight_lb','wingspan_in','standing_reach_in','vertical_max_in',
                  'vertical_standing_in','sprint_sec','agility_sec','bench_reps','body_fat_pct',
                  'ncaa_match_status','athlete_id','athlete_display_name',*ncaa_columns]
    feature_cols=list(dict.fromkeys(c for c in feature_cols if c in merged))
    features=merged[feature_cols].rename(columns={'athlete_id':'ncaa_espn_athlete_id',
                                                   'athlete_display_name':'ncaa_player_name'}).copy()
    features.sort_values('player_id').to_csv(F,index=False)
    outcomes=nba[['player_id','combine_year','total_games_3yr','total_points_3yr','ppg_3yr']].copy()
    outcomes.sort_values('player_id').to_csv(Y,index=False)
    audit=merged[['player_id','player_name','combine_year','ncaa_match_status','athlete_display_name','ncaa_games']].copy()
    audit.to_csv(DATA/'ncaa_match_audit.csv',index=False)
    print('NBA rows:',len(nba))
    print('College matching results:\n',features.ncaa_match_status.value_counts().to_string())
    print('Saved:', F, Y, 'and match audit')

features = pd.read_csv(F)
outcomes = pd.read_csv(Y)
print('Predraft features:', features.shape, '| NBA outcomes:', outcomes.shape)
print('Match status:\n', features.ncaa_match_status.value_counts().to_string())
print('Model-ready matched players:',features.ncaa_match_status.eq('matched').sum())
assert features.player_id.is_unique and outcomes.player_id.is_unique

NBA rows: 847
College matching results:
 ncaa_match_status
matched                        746
unmatched_or_ambiguous_ncaa    101
Saved: /Users/ishaantibdewal/Downloads/NBA_NCAA_COGS109_Simple_Package/data/predraft_features.csv /Users/ishaantibdewal/Downloads/NBA_NCAA_COGS109_Simple_Package/data/nba_outcomes.csv and match audit
Predraft features: (847, 28) | NBA outcomes: (847, 5)
Match status:
 ncaa_match_status
matched                        746
unmatched_or_ambiguous_ncaa    101
Model-ready matched players: 746


### Before modeling

Inspect the matched names in `data/ncaa_match_audit.csv` and check the match count. Because ESPN player IDs and NBA player IDs do **not** share a universal crosswalk, some legitimate players will be missed by strict name/year matching. Never automatically choose a fuzzy match. If the matched sample is too small for three models, consider expanding identification carefully or revisiting scope.

Your models are **scoring among NBA players with matched NCAA records**, not a model of which college players will make the NBA. **Do not use** future NBA points, games, or PPG as predictors.